# 🗂️ Notebook 2: Notification System — Data Model & APIs

## 🛠️ Setup

```bash
cd 06-system-designs/notification-system
uv sync
```

Select the `.venv` kernel in VS Code (top-right). If it doesn't appear, reload the window: `Cmd+Shift+P` → **Reload Window**.


## Data model

| Table | Columns |
|---|---|
| `templates` | id, name, body_template, channel |
| `prefs` | user_id, channel, enabled, quiet_hours |
| `send_log` | id, user_id, channel, idempotency_key, status, ts |
| `device_tokens` | user_id, platform, token |

### Idempotency

Caller includes a `dedup_key` (e.g., `order-123-shipped`). We store it in `send_log`;
a second attempt with the same key is a no-op success.

## API

```http
POST /notify
{
  "user_id": 42,
  "template": "order_shipped",
  "channel": "push",
  "priority": "normal",
  "dedup_key": "order-123-shipped",
  "vars": { "order_id": 123 }
}
```


In [ ]:
from pydantic import BaseModel
from typing import Literal

class NotifyRequest(BaseModel):
    user_id: int
    template: str
    channel: Literal["push","email","sms"]
    priority: Literal["high","normal","low"] = "normal"
    dedup_key: str
    vars: dict

print(NotifyRequest(user_id=42, template="order_shipped",
                    channel="push", dedup_key="k1", vars={"order_id":1}).model_dump_json(indent=2))
